In [1]:
# Calculate global mortality - requires ~100GB memory

In [2]:
import os
import xarray as xr
import warnings
from utils.utils import get_scenario_config
from utils.mortality_utils import mortality

In [3]:
# === Health variables ===
# COPD, DIABETES, ISCHEMIC_HEART_DISEASE, LOWER_RESPIRATORY_INFECTIONS, LUNG_CANCER, STROKE
# resp_copd, t2_dm, cvd_ihd, lri, neo_lung, cvd_stroke
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER", "STROKE"]

In [4]:
# === Path config ===
MASKS_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
BMR_DIR = "/glade/derecho/scratch/awells/air_quality/BMR/"
TMREL_DIR = "/glade/derecho/scratch/awells/air_quality/TMREL/"
RR_DIR = "/glade/derecho/scratch/awells/air_quality/rr_pm25/"

In [5]:
# Load country masks
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

# Load population file
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
pop = xr.open_dataarray(pop_path)
pop = pop.reindex_like(masks, method="nearest", tolerance=1e-9)

In [6]:
# === Calculate the scalar distributions ===
n_samples = 200

# TMREL from GBD21 (uniform distribution)
tmrel_file = f"TMREL_{n_samples}_samples_pm25.nc"
tmrel_path = os.path.join(TMREL_DIR, tmrel_file)
tmrel_da = xr.open_dataarray(tmrel_path)

In [7]:
warnings.filterwarnings('ignore')

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]
dates = f"{years.start}-{years.stop}"

PM25_DIR = f"/glade/work/awells/air_quality/{model}/pm25/annual_pm25_bc/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/pm25/global/{n_samples}_samples"

for health_VAR in health_vars:
    print(f"Processing health variable {health_VAR}")

    # RR from GBD21 (normal distribution)
    rr_file = f"RR_{health_VAR}_{n_samples}_samples_pm25.nc"
    rr_path = os.path.join(RR_DIR, rr_file)
    rr_da = xr.open_dataarray(rr_path)

    del rr_file, rr_path

    # Load BMR for each grid point
    bmr_file = f"GBD_BMR_Country_Mask_{health_VAR}_{n_samples}_samples_1990-2009.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    BMR = xr.open_dataarray(bmr_path)  # three quantiles

    del bmr_file, bmr_path

    for ens_num in ensemble_members:
        print(f"Processing ensemble member {ens_num:02d}")

        # Load ozone data
        pm25_file = f"Annual_PM25_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
        pm25_path = os.path.join(PM25_DIR, pm25_file)
        pm25 = xr.open_dataarray(pm25_path).astype("float32")

        # Adjust indices to match (with small tolerance)
        # e.g., max 1e-7 km distance
        pm25 = pm25.reindex_like(masks, method="nearest", tolerance=1e-9, fill_value=0)

        del pm25_file, pm25_path

        # make pm25 dask-backed
        pm25 = pm25.chunk({'lat': 180, 'lon': 360})

        for year in range(years.start, years.stop + 1):
            print(f"Processing year {year}")

            # Find RR at each grid point
            pm25_year = pm25.sel(year=year)
            RR = rr_da.interp(exposure=pm25_year)

            del pm25_year

            RR = RR.where(RR["exposure"] >= tmrel_da, 1)
            AF = (1 - (1/RR)).chunk({"samples": 10})

            del RR

            POP = pop.sel(year=year).chunk({"lat": 180, "lon": 360})

            M = mortality(AF, BMR, POP)

            del AF, POP

            global_M = M.sum(dim=("lat", "lon"))

            del M

            description = (f"Global {health_VAR} mortality due to PM2.5 "
                           "- scripts by A.F. Wells (2025)")
            global_M.attrs["description"] = description
            global_M.attrs["health_var"] = health_VAR
            global_M.attrs["model"] = model
            global_M.attrs["scenario"] = scenario
            global_M.attrs["ensemble_number"] = ens_num
            global_M.attrs["year"] = year

            out_file = f"Global_mortality_{health_VAR}_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{year}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)
            print(f"Saving to {out_path}")
            global_M.to_netcdf(out_path)

            del global_M

        del pm25

    del rr_da, BMR

print("All processing complete.")

Processing health variable COPD
Processing ensemble member 01
Processing year 2020
Saving to /glade/work/awells/air_quality/CESM2/mortality/pm25/global/200_samples/Global_mortality_COPD_200samples_CESM2_SSP245_G6_01_2020.nc
Processing year 2021
Saving to /glade/work/awells/air_quality/CESM2/mortality/pm25/global/200_samples/Global_mortality_COPD_200samples_CESM2_SSP245_G6_01_2021.nc
Processing year 2022
Saving to /glade/work/awells/air_quality/CESM2/mortality/pm25/global/200_samples/Global_mortality_COPD_200samples_CESM2_SSP245_G6_01_2022.nc
Processing year 2023
Saving to /glade/work/awells/air_quality/CESM2/mortality/pm25/global/200_samples/Global_mortality_COPD_200samples_CESM2_SSP245_G6_01_2023.nc
Processing year 2024
Saving to /glade/work/awells/air_quality/CESM2/mortality/pm25/global/200_samples/Global_mortality_COPD_200samples_CESM2_SSP245_G6_01_2024.nc
Processing year 2025
Saving to /glade/work/awells/air_quality/CESM2/mortality/pm25/global/200_samples/Global_mortality_COPD_200s